In [1]:
import pandas as pd
import numpy as np
import os

# 1. 定义数据路径（注意：从 notebooks 目录访问 data 需要用 ../ 向上返回一层）
BASE_DIR = "../data/raw"
orders_path = os.path.join(BASE_DIR, "olist_orders_dataset.csv")
payments_path = os.path.join(BASE_DIR, "olist_order_payments_dataset.csv")

# 2. 读取数据
df_orders = pd.read_csv(orders_path)
df_payments = pd.read_csv(payments_path)

print(f"订单表形状 (Rows, Cols): {df_orders.shape}")
print(f"支付表形状 (Rows, Cols): {df_payments.shape}")

订单表形状 (Rows, Cols): (99441, 8)
支付表形状 (Rows, Cols): (103886, 5)


In [2]:
# 1. 统计订单表中每一列的缺失值数量
print("=== 订单表缺失值统计 ===")
print(df_orders.isnull().sum())

# 2. 检查每一列的数据类型
print("\n=== 订单表字段类型 ===")
print(df_orders.dtypes)

=== 订单表缺失值统计 ===
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

=== 订单表字段类型 ===
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object


In [3]:
# 打印订单表的前 3 行
df_orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [5]:
df_clean = df_orders.copy()
time_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in time_cols:
    df_clean[col] = pd.to_datetime(df_clean[col])

print(df_clean[time_cols].dtypes)

time_anomaly_mask = df_clean['order_approved_at'] < df_clean['order_purchase_timestamp']
anomaly_count = time_anomaly_mask.sum()
print(anomaly_count)

if anomaly_count > 0:
    print(df_clean[time_anomaly_mask][['order_id', 'order_purchase_timestamp', 'order_approved_at']].head())

order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object
0


In [6]:
print(df_orders.isnull().mean())
print(df_payments.isnull().mean())

print(df_orders.duplicated().sum())
print(df_orders.duplicated(subset=['order_id']).sum())
print(df_payments.duplicated().sum())

print(df_payments.describe())

print(df_orders['order_status'].value_counts())
print(df_payments['payment_type'].value_counts())

print(df_orders.select_dtypes(include=['object']).mode().iloc[0])

order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.001609
order_delivered_carrier_date     0.017930
order_delivered_customer_date    0.029817
order_estimated_delivery_date    0.000000
dtype: float64
order_id                0.0
payment_sequential      0.0
payment_type            0.0
payment_installments    0.0
payment_value           0.0
dtype: float64
0
0
0
       payment_sequential  payment_installments  payment_value
count       103886.000000         103886.000000  103886.000000
mean             1.092679              2.853349     154.100380
std              0.706584              2.687051     217.494064
min              1.000000              0.000000       0.000000
25%              1.000000              1.000000      56.790000
50%              1.000000              1.000000     100.000000
75%              1.000000              4.000000

C:\Users\MajorTom\AppData\Local\Temp\ipykernel_29380\3133760887.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(df_orders.select_dtypes(include=['object']).mode().iloc[0])


order_id                         00010242fe8c5a6d1ba2dd792cb16214
customer_id                      00012a2ce6f8dcda20d059ce98491703
order_status                                            delivered
order_purchase_timestamp                      2017-11-20 10:59:08
order_approved_at                             2018-02-27 04:31:10
order_delivered_carrier_date                  2018-05-09 15:48:00
order_delivered_customer_date                 2016-10-27 17:32:07
order_estimated_delivery_date                 2017-12-20 00:00:00
Name: 0, dtype: str


In [7]:
anomaly_status_1 = df_clean[(df_clean['order_status'] == 'delivered') & (df_clean['order_delivered_customer_date'].isnull())]
print(len(anomaly_status_1))

anomaly_status_2 = df_clean[(df_clean['order_status'] != 'delivered') & (df_clean['order_delivered_customer_date'].notnull())]
print(len(anomaly_status_2))

anomaly_timeline_1 = df_clean[df_clean['order_delivered_customer_date'] < df_clean['order_delivered_carrier_date']]
print(len(anomaly_timeline_1))

anomaly_timeline_2 = df_clean[df_clean['order_delivered_carrier_date'] < df_clean['order_approved_at']]
print(len(anomaly_timeline_2))

8
6
23
1359


In [11]:
df_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

item_total = df_items.groupby('order_id')[['price', 'freight_value']].sum().sum(axis=1)
pay_total = df_payments.groupby('order_id')['payment_value'].sum()

financial_check = pd.merge(item_total.to_frame('item_amt'), pay_total.to_frame('pay_amt'), on='order_id')
financial_anomalies = financial_check[np.abs(financial_check['item_amt'] - financial_check['pay_amt']) > 0.05]
print(len(financial_anomalies))

delivery_duration = (df_clean['order_delivered_customer_date'] - df_clean['order_purchase_timestamp']).dt.days
outlier_delivery = df_clean[delivery_duration > 180]
print(len(outlier_delivery))

payment_sequence_dup = df_payments.duplicated(subset=['order_id', 'payment_sequential']).sum()
print(payment_sequence_dup)

260
14
0


In [12]:
import sys
import os
sys.path.append('../')
from src.cleaner import OlistDataCleaner

df_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

cleaner = OlistDataCleaner(df_orders, df_payments, df_items)
df_orders_clean, df_payments_clean, quality_report = cleaner.run_pipeline()

print(quality_report)

os.makedirs("../data/processed", exist_ok=True)
# 显式指定使用 fastparquet 引擎保存
df_orders_clean.to_parquet("../data/processed/olist_orders_clean.parquet", index=False, engine='fastparquet')
df_payments_clean.to_parquet("../data/processed/olist_payments_clean.parquet", index=False, engine='fastparquet')

{'raw_orders_count': 99441, 'raw_payments_count': 103886, 'status_delivered_missing_date': 8, 'status_not_delivered_has_date': 6, 'timeline_reverse_carrier_customer': 23, 'timeline_reverse_approved_carrier': 1359, 'outlier_delivery_duration': 14, 'financial_amount_discrepancy': 260, 'clean_orders_count': 97778, 'clean_payments_count': 102145}
